In [1]:
import json
from typing import Dict, Set, List
from functools import partial

import pandas as pd
import yaml
from IPython.display import display
from rapidfuzz import fuzz

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

psg_directory = "../data/geography/"
psg_data_file = "psgc_2026-01-13.csv"

In [2]:
df = pd.read_csv(psg_directory + psg_data_file)
display(df.info())
display(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43769 entries, 0 to 43768
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   psgc_id                43769 non-null  int64  
 1   name                   43769 non-null  object 
 2   correspondence_code    43719 non-null  float64
 3   geographic_level       43767 non-null  object 
 4   old_names              1699 non-null   object 
 5   city_class             149 non-null    object 
 6   income_classification  1724 non-null   object 
 7   settlement_type        42011 non-null  object 
 8   population             43768 non-null  object 
 9   Unnamed: 9             18 non-null     object 
 10  barangay_status        2855 non-null   object 
dtypes: float64(1), int64(1), object(9)
memory usage: 3.7+ MB


None

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status
0,1300000000,National Capital Region (NCR),130000000.0,Reg,NaN,NaN,NaN,NaN,"14,001,751",NaN,NaN
1,1380100000,City of Caloocan,137501000.0,City,NaN,HUC,1st,NaN,"1,712,945",NaN,NaN
2,1380100001,Barangay 1,137501001.0,Bgy,NaN,NaN,NaN,U,"2,356",NaN,NaN
3,1380100002,Barangay 2,137501002.0,Bgy,NaN,NaN,NaN,U,"5,226",NaN,NaN
4,1380100003,Barangay 3,137501003.0,Bgy,NaN,NaN,NaN,U,"2,544",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
43764,1999908006,Manaulanan,124712037.0,Bgy,NaN,NaN,NaN,U,"7,790",NaN,NaN
43765,1999908007,Pamalian,124712062.0,Bgy,NaN,NaN,NaN,R,"2,981",NaN,NaN
43766,1999908008,Tapodoc,124717017.0,Bgy,NaN,NaN,NaN,R,"1,797",NaN,NaN
43767,1999908009,Macabual,124712034.0,Bgy,NaN,NaN,NaN,R,"4,311",NaN,NaN


In [3]:
# this code is just copied from my barangay project so there are more explanations there
# i think

df["psgc_id"] = df["psgc_id"].astype(str).str.zfill(10)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

geographic_level_map = {
    "Reg": "region",
    "City": "city",
    "Mun": "municipality",
    "Prov": "province",
    "SubMun": "submunicipality",
    "Bgy": "barangay",
}
df["geographic_level"] = df["geographic_level"].replace(geographic_level_map)

df["barangay_code"] = df["psgc_id"].str[-3:]
df["municipal_or_city_code"] = df["psgc_id"].str[-5:-3]
df["province_or_huc_code"] = df["psgc_id"].str[-8:-5]
df["region_code"] = df["psgc_id"].str[-10:-8]

df["barangay_mapper"] = df["psgc_id"].str[-10:]
df["municipal_or_city_mapper"] = df["psgc_id"].str[-10:-3]
df["province_or_huc_mapper"] = df["psgc_id"].str[-10:-5]
df["region_mapper"] = df["psgc_id"].str[-10:-8]

df.sample(10)

regions_filter = (
    (df["province_or_huc_code"] == "000")
    & (df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)
regions_mapper = (
    df.loc[regions_filter, ["region_mapper", "name"]]
    .sort_values("region_mapper")
    .set_index("region_mapper", drop=True)
    .to_dict()["name"]
)


province_or_huc_filter = (
    ~(df["province_or_huc_code"] == "000")
    & (df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)

province_or_huc_mapper = (
    df.loc[province_or_huc_filter, ["province_or_huc_mapper", "name"]]
    .sort_values("province_or_huc_mapper")
    .set_index("province_or_huc_mapper")
    .to_dict()["name"]
)
municipal_or_city_filter = (
    ~(df["province_or_huc_code"] == "000")
    & ~(df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)

municipal_or_city_mapper = (
    df.loc[municipal_or_city_filter, ["municipal_or_city_mapper", "name"]]
    .sort_values("municipal_or_city_mapper")
    .set_index("municipal_or_city_mapper")
    .to_dict()["name"]
)

df["region"] = df["region_mapper"].map(regions_mapper)
df["province_or_huc"] = df["province_or_huc_mapper"].map(province_or_huc_mapper)
df["municipality_or_city"] = df["municipal_or_city_mapper"].map(
    municipal_or_city_mapper
)
display(df.sample(10))

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status,barangay_code,municipal_or_city_code,province_or_huc_code,region_code,barangay_mapper,municipal_or_city_mapper,province_or_huc_mapper,region_mapper,region,province_or_huc,municipality_or_city
14246,0403412002,Barangay Zone I,43412002.0,barangay,NaN,NaN,NaN,R,597,NaN,Pob.,002,12,034,04,0403412002,0403412,04034,04,Region IV-A (CALABARZON),Laguna,Luisiana
41862,1903605030,Tambac,153605030.0,barangay,NaN,NaN,NaN,R,934,NaN,NaN,030,05,036,19,1903605030,1903605,19036,19,Bangsamoro Autonomous Region In Muslim Mindana...,Lanao del Sur,Binidayan
40748,1606715009,Masgad,166715009.0,barangay,NaN,NaN,NaN,R,"2,242",NaN,NaN,009,15,067,16,1606715009,1606715,16067,16,Region XIII (Caraga),Surigao del Norte,Malimono
26650,0701220007,Bauhugan,71220007.0,barangay,NaN,NaN,NaN,R,344,NaN,NaN,007,20,012,07,0701220007,0701220,07012,07,Region VII (Central Visayas),Bohol,Dimiao
32217,0806021020,Miramar,86021020.0,barangay,NaN,NaN,NaN,R,336,NaN,Pob.,020,21,060,08,0806021020,0806021,08060,08,Region VIII (Eastern Visayas),Samar,Villareal
15794,0405646014,Katimo,45646014.0,barangay,NaN,NaN,NaN,R,"1,970",NaN,NaN,014,46,056,04,0405646014,0405646,04056,04,Region IV-A (CALABARZON),Quezon,Tagkawayan
36079,1001801001,Alga,101801001.0,barangay,NaN,NaN,NaN,R,816,NaN,NaN,001,01,018,10,1001801001,1001801,10018,10,Region X (Northern Mindanao),Camiguin,Catarman
35677,1001305006,Guihean,101305006.0,barangay,NaN,NaN,NaN,R,"2,530",NaN,NaN,006,05,013,10,1001305006,1001305,10013,10,Region X (Northern Mindanao),Bukidnon,Impasug-ong
750,1380606072,Barangay 466,133906072.0,barangay,NaN,NaN,NaN,U,"1,448",NaN,NaN,072,06,806,13,1380606072,1380606,13806,13,National Capital Region (NCR),City of Manila,Sampaloc
22043,0600612002,Amparo,60612002.0,barangay,NaN,NaN,NaN,R,949,NaN,NaN,002,12,006,06,0600612002,0600612,06006,06,Region VI (Western Visayas),Antique,Patnongon


In [4]:
def sanitize_input(
    input_str: str | None, exclude: List[str] | str | None = None
) -> str:
    """
    Removes whitespaces, lowers, and remove all strings listed in exclude. If
    data is incompatible, will coerce to empty string.
    """
    if input_str is None:
        input_str = ""
    if not isinstance(input_str, str):
        input_str = ""
    sanitized_str = input_str.lower()
    if exclude is None:
        return sanitized_str

    if isinstance(exclude, list):
        exclude = [x.lower() for x in exclude if isinstance(x, str)]
        for item in exclude:
            sanitized_str = sanitized_str.replace(item, "")
        return sanitized_str

    return sanitized_str.replace(exclude.lower(), "")


cleanerjim = partial(
    sanitize_input, exclude=["(pob.)", "(pob)", ".", " ", "-", "(", ")", "&", "pob."]
)



In [5]:
bdf = df[df["geographic_level"] == "barangay"]

In [6]:
fuzzer_base = bdf[["name", "province_or_huc", "municipality_or_city", "psgc_id"]]

In [7]:
fuzzer_base = fuzzer_base.rename({"name": "barangay"}, axis=1)
fuzzer_base.to_parquet("../data/geography/fuzzer_base.parquet")

In [8]:
fuzzer_base

,barangay,province_or_huc,municipality_or_city,psgc_id
2,Barangay 1,City of Caloocan,NaN,1380100001
3,Barangay 2,City of Caloocan,NaN,1380100002
4,Barangay 3,City of Caloocan,NaN,1380100003
5,Barangay 4,City of Caloocan,NaN,1380100004
6,Barangay 5,City of Caloocan,NaN,1380100005
...,...,...,...,...
43764,Manaulanan,Special Geographic Area,Tugunan,1999908006
43765,Pamalian,Special Geographic Area,Tugunan,1999908007
43766,Tapodoc,Special Geographic Area,Tugunan,1999908008
43767,Macabual,Special Geographic Area,Tugunan,1999908009


In [2]:
from __future__ import annotations
from typing import TypeAlias

TreeNode: TypeAlias = dict[str, "TreeNode"] | list[str]

In [3]:
from barangay import BARANGAY

In [ ]:
barangay: TreeNode = BARANGAY

barangay["Farm"] = "yes"


